<a href="https://colab.research.google.com/github/cy6erlizard/landsat-lake-clarity/blob/main/notebooks/02_select_lakes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 - Choose the target lakes, and measure the ceiling

Region: **Northern Lower Michigan** (clear glacial kettle lakes, same optical regime as the
New Hampshire reference lakes, with decades of Cooperative Lakes Monitoring Program Secchi).

**The gate.** One large lake and one small one, both with enough FIELD July-years to support a
client-style July-annual-mean validation. Selection is on distinct years with a July field
Secchi reading in the Water Quality Portal, NOT on coincident satellite/in-situ matchups.
Matchups require a clean pass within three days of a reading and badly undercount the
achievable validation sample; the client's own r = -0.22 for Squam used the full field record,
not matchups. If no small lake qualifies, `select_target_lakes` raises.

**The ceiling.** The intraclass correlation of log Secchi across the region's lakes: the
fraction of the pooled signal that says nothing about tracking one lake through time. It is the
number that reconciles a published R-squared of 0.637 with a per-lake r of -0.22.

In [1]:
# Cell 1 of 6: repo + Drive
import os, pathlib, subprocess, sys

REPO = "https://github.com/cy6erlizard/landsat-lake-clarity.git"
ROOT = pathlib.Path("/content/landsat-lake-clarity")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / "src"))  # make lakeclarity importable in the running kernel

from google.colab import drive
drive.mount("/content/drive")
os.environ["LAKECLARITY_DATA"] = "/content/drive/MyDrive/lake-clarity"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 2 of 6: training set (all Michigan) and target region (northern counties)
import pandas as pd
from lakeclarity import config, eda, edi, features, locus, selection, viz, wqp

viz.use_style()

matchups = edi.load_matchups()
lakes = locus.load_lakes()

# Piper, Glines & Rose trained on ALL Wisconsin lakes in AquaSat, not just their
# 127 study lakes. The equivalent is every Michigan lake. Target lakes come from
# the narrower northern-county region and are held out of training entirely.
train_lk = locus.train_lakes(lakes)
region_lk = locus.region_lakes(lakes)
print(f"{len(train_lk):,} {config.TRAIN_REGION_NAME} lakes, "
      f"{len(region_lk):,} {config.REGION_NAME} lakes in LOCUS")

region = matchups[matchups["lagoslakeid"].isin(train_lk["lagoslakeid"])].copy()
region = region[region[config.TARGET].notna()]
region = features.add_time_columns(region)
region = features.add_log_target(region)
print(f"{len(region):,} Secchi matchups on {region.lagoslakeid.nunique():,} lakes (training supply)")
region.to_parquet(config.INTERIM_DIR / "region_matchups.parquet", index=False)

In [3]:
# Cell 3 of 6: FIELD coverage from the Water Quality Portal (the selection metric)
# Confirm the characteristic-name choice, then pull field Secchi + station coords,
# map each station to its nearest lake, and count distinct July-years per lake.
print(wqp.characteristic_coverage().to_string(index=False), "\n")

field = wqp.fetch_secchi()                 # cached to Drive after first pull
stations = wqp.fetch_stations()            # site coordinates
site_to_lake = wqp.map_sites_to_lakes(stations, lakes)
field_cov = wqp.lake_field_coverage(field, site_to_lake)

site_to_lake.to_parquet(config.INTERIM_DIR / "wqp_site_to_lake.parquet", index=False)
field_cov.to_csv(config.TABLE_DIR / "T04b_field_coverage.csv")
print(f"{len(field):,} field Secchi records, {len(stations):,} stations, "
      f"{len(site_to_lake):,} mapped to a lake")
print(f"lakes with >= {config.MIN_FIELD_JULY_YEARS} field July-years: "
      f"{(field_cov['field_july_years'] >= config.MIN_FIELD_JULY_YEARS).sum()}")
print(field_cov.head(12).to_string())

                 characteristic  n_records         status
       Depth, Secchi disk depth      61512             ok
  Secchi Reservoir Transparency          0 rejected (400)
Water transparency, Secchi disc          0             ok 

60,809 field Secchi records, 941 stations, 790 mapped to a lake
lakes with >= 15 field July-years: 56
             field_july_years  field_july_n  field_year_first  field_year_last  field_secchi_mean  field_all_years
lagoslakeid                                                                                                       
97859                      41           250              1984             2024           3.826627               41
2392                       39           336              1984             2023           2.914499               39
1141                       36           422              1984             2023           4.716940               38
1627                       35           165              1984             2019           

In [ ]:
# Cell 4 of 6: THE CEILING. Run this before choosing anything.
# A regional ICC means nothing on its own. The claim being tested is that the
# NATIONAL training distribution is dominated by between-lake variance, which a
# model can capture while having no within-lake skill anywhere. So compute both.
national = matchups[matchups[config.TARGET].notna()].copy()
national = features.add_log_target(national)

nat = selection.ceiling_report(national)
ceiling = selection.ceiling_report(region)

print(f"{'':>30} {'national (all US)':>18} {config.TRAIN_REGION_NAME:>18}")
for k in ["icc", "between_lake_sd_log10", "within_lake_sd_log10", "n_lakes", "n_obs"]:
    print(f"{k:>30} {nat[k]:>18,.3f} {ceiling[k]:>18,.3f}")

print(f"\nNational: {nat['pct_variance_between_lakes']:.0f}% of the variance is lakes differing")
print(f"from one another. That is the variance a pooled R2 of 0.637 is mostly measuring,")
print("and it says nothing about tracking one lake through time.")
print(f"\nWithin {config.TRAIN_REGION_NAME} the lakes resemble each other, so between-lake")
print(f"variance falls to {ceiling['pct_variance_between_lakes']:.0f}% and "
      f"{ceiling['pct_variance_within_lakes']:.0f}% of what remains is genuine temporal")
print("signal. That is the signal a regional model can learn and a national model drowns.")

pd.Series(ceiling).to_csv(config.TABLE_DIR / "T03_variance_ceiling.csv")
pd.DataFrame({"national": nat, "regional": ceiling}).to_csv(
    config.TABLE_DIR / "T03b_icc_national_vs_regional.csv")

In [5]:
# Cell 5 of 6: the gate
candidates = selection.candidate_table(region, region_lk, field_cov)
candidates.to_csv(config.TABLE_DIR / "T04_candidate_lakes.csv", index=False)
print(candidates.head(15)[["lagoslakeid", "lake_name", "lake_waterarea_ha",
                            "field_july_years", "field_year_first", "field_year_last",
                            "field_secchi_mean", "n_matchups"]].to_string(index=False))

try:
    targets = selection.select_target_lakes(candidates)
    print("\n" + targets.summary())
    pd.Series({"large": targets.ids[0], "small": targets.ids[1]}).to_csv(
        config.TABLE_DIR / "T05_target_lakes.csv")
except selection.NoEligibleLakeError as exc:
    print("\nGATE FAILED:", exc)
    print("\nDo not work around this. Either relax MIN_FIELD_JULY_YEARS with a written")
    print("justification, widen the region, or accept that the small-lake case cannot")
    print("be validated with the data that exists.")
    raise

 lagoslakeid        lake_name  lake_waterarea_ha  field_july_years  field_year_first  field_year_last  field_secchi_mean  n_matchups
     97859.0        Loon Lake          37.904011                41              1984             2024           3.826627          42
      2392.0              NaN                NaN                39              1984             2023           2.914499           0
      1141.0        Glen Lake        2546.573792                36              1984             2023           4.716940           9
      1627.0      Platte Lake        1025.222497                35              1984             2019           3.869121         212
      1362.0     Higgins Lake        4126.624360                34              1984             2023           9.952173          26
      3270.0     Arbutus Lake         153.210846                34              1988             2023           4.398545           9
    138062.0              NaN                NaN                34   

In [ ]:
# Cell 6 of 6: figures F6 to F9
for fig, fid, slug in [
    (selection.fig_region_map(candidates, targets), "F06", "region_map"),
    (selection.fig_area_vs_pixels(candidates, targets), "F07", "area_vs_pixels"),
    (selection.fig_variance_decomposition(region, national=national), "F08", "variance_decomposition"),
    (selection.fig_candidate_timeseries(field, site_to_lake, candidates), "F09", "candidate_timeseries"),
]:
    print("wrote", viz.save(fig, fid, slug).name)